In [1]:
import numpy as np
from sklearn import preprocessing
import os
print("Current working directory:", os.getcwd())
file_path = 'C:/Users/cross/Downloads/Audiobooks_data.csv'
raw_csv_data = np.loadtxt(file_path, delimiter = ',')
unscaled_inputs_all = raw_csv_data[:,1:-1]
targets_all = raw_csv_data[:,-1]

Current working directory: C:\Users\cross\Downloads


In [2]:
num_one_targets = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove = []
for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter>num_one_targets:
            indices_to_remove.append(i)
unscaled_inputs_equals_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis = 0)
targets_equals_priors = np.delete(targets_all, indices_to_remove, axis = 0)

In [3]:
scaled_inputs = preprocessing.scale(unscaled_inputs_equals_priors)

In [4]:
shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)
shuffled_inputs = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equals_priors[shuffled_indices]

In [5]:
samples_count = shuffled_inputs.shape[0]
train_samples_count = int(0.8* samples_count)
validation_samples_count = int(0.1* samples_count)
test_samples_count = samples_count - train_samples_count - validation_samples_count

train_inputs = shuffled_inputs[: train_samples_count]
train_targets = shuffled_targets[: train_samples_count]

validation_inputs = shuffled_inputs[train_samples_count: train_samples_count + validation_samples_count]
validation_targets = shuffled_targets[train_samples_count: train_samples_count + validation_samples_count]

test_inputs = shuffled_inputs[train_samples_count + validation_samples_count:]
test_targets = shuffled_targets[train_samples_count + validation_samples_count:]

print(np.sum(train_targets), train_samples_count, np.sum(train_targets)/train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets)/validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets)/test_samples_count)


1812.0 3579 0.5062866722548198
208.0 447 0.465324384787472
217.0 448 0.484375


In [6]:
np.savez('Audiobooks_data_train', inputs = train_inputs, targets = train_targets)
np.savez('Audiobooks_data_validation', inputs = validation_inputs, targets = validation_targets)
np.savez('Audiobooks_data_test', inputs = test_inputs, targets = test_targets)

In [7]:
import numpy as np
import tensorflow as tf

npz = np.load('Audiobooks_data_train.npz')
train_inputs, train_targets = npz['inputs'].astype(np.float64), npz['targets'].astype(np.int64)
npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float64), npz['targets'].astype(np.int64)
npz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float64), npz['targets'].astype(np.int64)

In [14]:
input_size = 10
output_size = 2
hidden_layer_size = 50

model = tf.keras.Sequential([
                            tf.keras.layers.Dense(hidden_layer_size, activation = 'relu'),
                            tf.keras.layers.Dense(hidden_layer_size, activation = 'relu'),
                            tf.keras.layers.Dense(output_size, activation = 'softmax')
                            ])
model.compile(optimizer= 'adam',loss = 'sparse_categorical_crossentropy', metrics=['accuracy'])
early_stopping = tf.keras.callbacks.EarlyStopping( patience = 2)

batch_size = 100
max_epochs = 100
model.fit(train_inputs, train_targets, batch_size = batch_size, epochs = max_epochs, callbacks = [early_stopping], validation_data = (validation_inputs, validation_targets), verbose = 2)

Epoch 1/100
36/36 - 1s - 39ms/step - accuracy: 0.6832 - loss: 0.5829 - val_accuracy: 0.7427 - val_loss: 0.4927
Epoch 2/100
36/36 - 0s - 4ms/step - accuracy: 0.7625 - loss: 0.4684 - val_accuracy: 0.7852 - val_loss: 0.4310
Epoch 3/100
36/36 - 0s - 5ms/step - accuracy: 0.7748 - loss: 0.4220 - val_accuracy: 0.7942 - val_loss: 0.3978
Epoch 4/100
36/36 - 0s - 4ms/step - accuracy: 0.7896 - loss: 0.3946 - val_accuracy: 0.8076 - val_loss: 0.3822
Epoch 5/100
36/36 - 0s - 4ms/step - accuracy: 0.8044 - loss: 0.3784 - val_accuracy: 0.8389 - val_loss: 0.3576
Epoch 6/100
36/36 - 0s - 4ms/step - accuracy: 0.8025 - loss: 0.3677 - val_accuracy: 0.7942 - val_loss: 0.3730
Epoch 7/100
36/36 - 0s - 10ms/step - accuracy: 0.8131 - loss: 0.3585 - val_accuracy: 0.8233 - val_loss: 0.3442
Epoch 8/100
36/36 - 0s - 4ms/step - accuracy: 0.8108 - loss: 0.3527 - val_accuracy: 0.8098 - val_loss: 0.3455
Epoch 9/100
36/36 - 0s - 4ms/step - accuracy: 0.8108 - loss: 0.3485 - val_accuracy: 0.8210 - val_loss: 0.3388
Epoch 10